# Image Captioning - Training & Evaluation

This notebook trains and evaluates the model using `modelv3.py`.

In [ ]:
import os
import json
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

from dataloader import get_dataloaders, get_test_dataloader
from modelv3 import ImageCaptioningModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
config = {
    'vocab_path': 'vocab.json',
    'checkpoint_dir': 'checkpointsv3',
    'embed_dim': 256,
    'num_heads': 4,
    'num_layers': 2,
    'ff_dim': 512,
    'max_len': 128,
    'dropout': 0.3,
    'encoder_lr': 1e-4,
    'decoder_lr': 3e-4,
    'weight_decay': 1e-4,
    'epochs': 20,
    'batch_size': 32,
    'patience': 5,
    'unfreeze_last_epoch': 8,
    'layer_lr': 1e-5,
    'annotations_path': os.path.join('..', 'data', 'raw', 'tiny-trcap-en.json'),
    'image_dir': os.path.join('..', 'data', 'raw', 'images'),
    'split_ratios': (0.70, 0.10, 0.20),
    'caption_mode': 'random',
    'augment': True,
    'num_workers': 2,
    'seed': 42,
}

os.makedirs(config['checkpoint_dir'], exist_ok=True)

In [ ]:
with open(config['vocab_path'], 'r') as f:
    vocab = json.load(f)

vocab_size = len(vocab)
pad_idx = vocab.get('<PAD>', 0)

train_loader, val_loader = get_dataloaders(config)
test_loader = get_test_dataloader(config)
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")

In [ ]:
model = ImageCaptioningModel(
    vocab_size=vocab_size,
    embed_dim=config['embed_dim'],
    num_heads=config['num_heads'],
    num_layers=config['num_layers'],
    ff_dim=config['ff_dim'],
    max_len=config['max_len'],
    dropout=config['dropout']
).to(device)

encoder_params = (
    list(model.encoder.project.parameters())
    + list(model.encoder.pos_emb.parameters())
    + list(model.encoder.norm.parameters())
)
decoder_params = list(model.decoder.parameters())

optimizer = AdamW([
    {'params': encoder_params, 'lr': config['encoder_lr']},
    {'params': decoder_params, 'lr': config['decoder_lr']},
], weight_decay=config['weight_decay'])

scheduler = CosineAnnealingLR(optimizer, T_max=config['epochs'], eta_min=1e-6)
criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)

In [ ]:
def train_one_epoch(model, dataloader, optimizer, criterion, device, pad_idx):
    model.train()
    model.encoder.backbone.eval() # Keep backbone mostly eval (batchnorm fixed)
    if model.encoder.last_layers_unfrozen:
        model.encoder.backbone[-1].train()

    total_loss = 0
    total_tokens = 0

    for batch in dataloader:
        if batch is None: continue
        images = batch['image'].to(device)
        captions = batch['caption'].to(device)

        input_captions = captions[:, :-1]
        target_captions = captions[:, 1:]
        pad_mask = (input_captions == pad_idx)

        optimizer.zero_grad()
        logits = model(images, input_captions, tgt_key_padding_mask=pad_mask)

        logits_flat = logits.reshape(-1, logits.size(-1))
        targets_flat = target_captions.reshape(-1)
        loss = criterion(logits_flat, targets_flat)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        non_pad = (targets_flat != pad_idx).sum().item()
        total_loss += loss.item() * non_pad
        total_tokens += non_pad

    return total_loss / max(total_tokens, 1)

def validate(model, dataloader, criterion, device, pad_idx):
    model.eval()
    total_loss = 0
    total_tokens = 0
    with torch.no_grad():
        for batch in dataloader:
            if batch is None: continue
            images = batch['image'].to(device)
            captions = batch['caption'].to(device)
            input_captions = captions[:, :-1]
            target_captions = captions[:, 1:]
            pad_mask = (input_captions == pad_idx)

            logits = model(images, input_captions, tgt_key_padding_mask=pad_mask)
            logits_flat = logits.reshape(-1, logits.size(-1))
            targets_flat = target_captions.reshape(-1)
            loss = criterion(logits_flat, targets_flat)

            non_pad = (targets_flat != pad_idx).sum().item()
            total_loss += loss.item() * non_pad
            total_tokens += non_pad
    return total_loss / max(total_tokens, 1)

In [ ]:
best_val_loss = float('inf')
patience_counter = 0
history = []

print("Starting training...")
for epoch in range(1, config['epochs'] + 1):
    if epoch == config['unfreeze_last_epoch'] and not model.encoder.last_layers_unfrozen:
        print("Unfreezing last layers of the backbone...")
        model.encoder.unfreeze_last_layers()
        unfrozen_params = [p for p in model.encoder.backbone[-1].parameters() if p.requires_grad]
        optimizer.add_param_group({'params': unfrozen_params, 'lr': config['layer_lr'], 'weight_decay': config['weight_decay']})

    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device, pad_idx)
    val_loss = validate(model, val_loader, criterion, device, pad_idx)
    scheduler.step()

    print(f"Epoch {epoch}/{config['epochs']} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    history.append({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss})

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save({'model_state_dict': model.state_dict()}, os.path.join(config['checkpoint_dir'], 'best_model.pt'))
        print("  -> Best model saved!")
    else:
        patience_counter += 1
        if patience_counter >= config['patience']:
            print(f"Early stopping triggered after {epoch} epochs.")
            break

with open(os.path.join(config['checkpoint_dir'], 'history.json'), 'w') as f:
    json.dump(history, f, indent=2)

In [ ]:
# Evaluation on Test Set
print("\nLoading best model for evaluation...")
best_ckpt = torch.load(os.path.join(config['checkpoint_dir'], 'best_model.pt'))
model.load_state_dict(best_ckpt['model_state_dict'])

test_loss = validate(model, test_loader, criterion, device, pad_idx)
print(f"Test Loss: {test_loss:.4f}")